# 01 — Data Loading & Filtering

Load raw SIH and CNES parquet files, filter to kidney stone (N20) admissions in São Paulo, validate schema, and save clean datasets for downstream notebooks.

In [1]:
from pathlib import Path
import pandas as pd
import warnings

from shared import OUTPUT_DIR, DATA_DIR

warnings.filterwarnings("ignore", category=FutureWarning)

PROJECT_ROOT = Path("../../..")
RAW_DATA_DIR = PROJECT_ROOT / "data"
DATA_DIR.mkdir(parents=True, exist_ok=True)

## Load SIH Data

SIH AIH Reduzida files are stored as partitioned parquets: `data/sih/RDSPYYMM.parquet/`.
Each directory contains the hospital admission records for one month.

In [2]:
SIH_COLS = [
    "DIAG_PRINC", "DIAG_SECUN", "PROC_REA", "PROC_SOLIC",
    "DIAS_PERM", "MUNIC_RES", "MUNIC_MOV", "CAR_INT",
    "ESPEC", "CNES", "IDADE", "COD_IDADE", "SEXO",
    "VAL_TOT", "MORTE", "DT_INTER", "DT_SAIDA",
    "MARCA_UTI", "COMPLEX", "NATUREZA", "UF_ZI",
]

sih_dir = RAW_DATA_DIR / "sih"
sih_files = sorted(sih_dir.glob("RDSP*.parquet"))
print(f"Found {len(sih_files)} SIH parquet directories")
print(f"Date range: {sih_files[0].name} to {sih_files[-1].name}")

Found 120 SIH parquet directories
Date range: RDSP1601.parquet to RDSP2512.parquet


In [3]:
frames = []
for f in sih_files:
    df = pd.read_parquet(f)
    available = [c for c in SIH_COLS if c in df.columns]
    frames.append(df[available])

sih = pd.concat(frames, ignore_index=True)
print(f"Total SIH records: {len(sih):,}")
print(f"Columns loaded: {list(sih.columns)}")

Total SIH records: 25,787,169
Columns loaded: ['DIAG_PRINC', 'DIAG_SECUN', 'PROC_REA', 'PROC_SOLIC', 'DIAS_PERM', 'MUNIC_RES', 'MUNIC_MOV', 'CAR_INT', 'ESPEC', 'CNES', 'IDADE', 'COD_IDADE', 'SEXO', 'VAL_TOT', 'MORTE', 'DT_INTER', 'DT_SAIDA', 'MARCA_UTI', 'COMPLEX', 'NATUREZA', 'UF_ZI']


## Filter to Kidney Stone (N20)

In [4]:
sih["DIAG_PRINC"] = sih["DIAG_PRINC"].astype(str).str.strip()
kidney = sih[sih["DIAG_PRINC"].str.startswith("N20")].copy()
print(f"Kidney stone (N20) admissions: {len(kidney):,}")
print(f"Fraction of total: {len(kidney)/len(sih)*100:.2f}%")
print(f"\nSub-diagnoses:")
print(kidney["DIAG_PRINC"].value_counts())

Kidney stone (N20) admissions: 206,500
Fraction of total: 0.80%

Sub-diagnoses:
DIAG_PRINC
N201    80523
N200    72093
N202    39539
N209    12427
N20      1918
Name: count, dtype: int64


## Parse Dates and Validate

In [5]:
kidney["DT_INTER"] = pd.to_datetime(kidney["DT_INTER"], format="%Y%m%d", errors="coerce")
kidney["year"] = kidney["DT_INTER"].dt.year
kidney["month"] = kidney["DT_INTER"].dt.month

kidney["DIAS_PERM"] = pd.to_numeric(kidney["DIAS_PERM"], errors="coerce")
kidney["VAL_TOT"] = pd.to_numeric(kidney["VAL_TOT"], errors="coerce")
kidney["IDADE"] = pd.to_numeric(kidney["IDADE"], errors="coerce")

print(f"Year range: {kidney['year'].min()} to {kidney['year'].max()}")
print(f"Missing DIAS_PERM: {kidney['DIAS_PERM'].isna().sum()}")
print(f"Missing DT_INTER: {kidney['DT_INTER'].isna().sum()}")
print(f"\nBasic stats:")
kidney[["DIAS_PERM", "VAL_TOT", "IDADE"]].describe()

Year range: 2015 to 2025
Missing DIAS_PERM: 0
Missing DT_INTER: 0

Basic stats:


,DIAS_PERM,VAL_TOT,IDADE
count,206500.000000,206500.000000,206500.000000
mean,2.457458,909.555233,46.316596
std,3.140623,980.671059,15.265192
min,0.000000,0.000000,0.000000
25%,1.000000,414.680000,35.000000
50%,2.000000,766.110000,46.000000
75%,3.000000,1130.070000,57.000000
max,97.000000,69163.680000,98.000000


## Load CNES Facility Data

In [6]:
cnes_dir = RAW_DATA_DIR / "cnes"
cnes_files = sorted(cnes_dir.glob("STSP*.parquet"))
print(f"Found {len(cnes_files)} CNES establishment files")

# Load the most recent CNES snapshot for facility characteristics
if cnes_files:
    cnes = pd.read_parquet(cnes_files[-1])
    print(f"CNES records: {len(cnes):,}")
    print(f"Columns: {list(cnes.columns)[:20]}...")
else:
    print("WARNING: No CNES files found. Facility features will be unavailable.")
    cnes = pd.DataFrame()

Found 120 CNES establishment files
CNES records: 109,849
Columns: ['CNES', 'CODUFMUN', 'COD_CEP', 'CPF_CNPJ', 'PF_PJ', 'NIV_DEP', 'CNPJ_MAN', 'COD_IR', 'REGSAUDE', 'MICR_REG', 'DISTRSAN', 'DISTRADM', 'VINC_SUS', 'TPGESTAO', 'ESFERA_A', 'RETENCAO', 'ATIVIDAD', 'NATUREZA', 'CLIENTEL', 'TP_UNID']...


## Save Filtered Datasets

In [7]:
kidney.to_parquet(DATA_DIR / "kidney_sih.parquet", index=False)
if not cnes.empty:
    cnes.to_parquet(DATA_DIR / "kidney_cnes.parquet", index=False)

print(f"Saved kidney_sih.parquet: {len(kidney):,} rows")
print(f"Saved kidney_cnes.parquet: {len(cnes):,} rows")
print("\nData loading complete. Proceed to 02_exploratory.ipynb.")

Saved kidney_sih.parquet: 206,500 rows
Saved kidney_cnes.parquet: 109,849 rows

Data loading complete. Proceed to 02_exploratory.ipynb.
